## 炭素結晶構造



論文ではGRRM法により炭素８原子による構造の構造探索を行っています。
ここでは以下の説明変数を再度作成し、使用します。

***説明変数***

- Behlerの二体symmetry functionによる変換した量を説明変数とする。著作権のため構造は本ハンズオンに含まれません。

***目的変数***

- 全エネルギーをsiestaでPBE+D2で計算し直しています。このため論文の値と一致しません。

In [ ]:
import pandas as pd
df = pd.read_csv("../data_calculated/Carbon8_cell_descriptor_Etot.csv").set_index("key")
descriptor_names = ['a0.25_rp1.0', 'a0.25_rp1.5', 'a0.25_rp2.0', 'a0.25_rp2.5',
       'a0.25_rp3.0', 'a0.5_rp1.0', 'a0.5_rp1.5', 'a0.5_rp2.0', 'a0.5_rp2.5',
       'a0.5_rp3.0', 'a1.0_rp1.0', 'a1.0_rp1.5', 'a1.0_rp2.0', 'a1.0_rp2.5',
       'a1.0_rp3.0']
target_name = 'Etot'

In [ ]:
df

メタデータkeyとEtotの関係を図示します。

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
df.plot(y=target_name)
plt.ylabel("Etot (cell)")

1-abs(Pearsonの相間関数）を距離として説明変数・目的変数間の距離を示します。

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist,squareform
from scipy.cluster.hierarchy import dendrogram, linkage
import copy
from scipy.stats import pearsonr
import numpy as np

def make_linkage(df, descriptor_names, target_name, corr="minus_abs_pearson"):
    labels = copy.deepcopy(descriptor_names)
    labels.append(target_name)
    Xraw = df.loc[:,labels].values
    scaler = StandardScaler()
    X = scaler.fit_transform(Xraw)
    df_tmp = pd.DataFrame(X)
    if corr=="minus_abs_pearson":
        corr = 1- np.abs(df_tmp.corr())
    else:
        raise ValueError("unknown corr={}".format(corr))
    pairdistance = squareform(corr)
    Z = linkage(pairdistance)
    return Z, labels

def show_dendrogram(Z,labels, corr):
    fig, ax = plt.subplots()
    dendrogram(Z,labels=labels,orientation="left", ax=ax)
    ax.set_xlabel(corr)
    fig.tight_layout()
    fig.show()
    
corr="minus_abs_pearson"
Z, labels = make_linkage(df, descriptor_names, target_name, corr)
show_dendrogram(Z,labels, corr)

以下に代表的な構造を示します。
二次元構造の層間は綺麗にstackingしているわけではないですが、参考にgraphiteとgrapheneとして近いstackingを書いておきます。

- 3D-000 (graphite) (~AB stacking)
![](../data/Carbon8_image/3D-000.png)

- 3D-001 (diamond)
![](../data/Carbon8_image/3D-001.png)


- 3D-002 (graphite) (~AB stacking)
![](../data/Carbon8_image/3D-002.png)

- 3D-003 (hexagonal diamond)
![](../data/Carbon8_image/3D-003.png)

- 3D-004 (sp3結合で四員環を含む３次元構造)
![](../data/Carbon8_image/3D-004.png)

- 3D-005 (crossed graphene)
![](../data/Carbon8_image/3D-005.png)



- 2D-000 (4 layer graphete) (~ABAB stacking)
![](../data/Carbon8_image/2D-000.png)

- 2D-001 (4 layer graphete) (~ABAB stacking)
![](../data/Carbon8_image/2D-001.png)

- 2D-002 (4 layer graphete) (~AABA stacking)
![](../data/Carbon8_image/2D-002.png)

- 2D-003 (bilayer graphete) (~AB stacking)
![](../data/Carbon8_image/2D-003.png)

- 2D-004 (mono layer graphete)
![](../data/Carbon8_image/2D-004.png)

- 2D-005 (mono layer graphete)
![](../data/Carbon8_image/2D-005.png)


In [ ]:
import os
import seaborn as sns

IMAGE_DIR = "image_keep"

imgfile = os.path.join(IMAGE_DIR, "carbon_pairplot.png")
if not os.path.isfile(imgfile):
    img = sns.pairplot(df)
    os.makedirs(IMAGE_DIR, exist_ok=True)
    img.savefig(imgfile)
    
from IPython import display
display.Image(imgfile)

**参考文献**

1. Makito Takagi, Tetsuya Taketsugu, Hiori Kino, Yoshitaka Tateyama, Kiyoyuki Terakura, and Satoshi Maeda,
"Global search for low-lying crystal structures using the artificial force induced reaction method: A case study on carbon",
Phys. Rev. B 95, 184110, (2017)

2. Jörg Behler and Michele Parrinello, \"Generalized Neural-Network Representation of High-Dimensional Potential-Energy Surfaces\", Phys. Rev. Lett. 98, 146401(2007)